# Contrail segmentation — U-Net + EfficientNet-B2 · V2

**Dataset:** GVCCS · Zenodo 15743988 · CC BY 4.0 · EUROCONTROL MUAC  
**Model:** U-Net + EfficientNet-B2 (ImageNet pretrained)  
**Estimated time:** ~7 h on Kaggle T4 x2  

### Changes vs V1 (B4 run)
| What | V1 | V2 |
|---|---|---|
| Encoder | EfficientNet-B4 (17M params) | **EfficientNet-B2 (7M params)** — less overfitting |
| Loss | Dice + BCE | **Dice + Focal Loss** (γ=2, α=0.25) — handles thin structures |
| Augmentation | Flip + brightness | **+ Rotate90 + ShiftScaleRotate + ElasticTransform + CoarseDropout** |
| Epochs | 30 | **40** (B2 is ~35% faster per epoch) |
| Checkpoint saves | Output tab only | **HuggingFace Hub after every epoch** |

> ⚠️ B4 weights are NOT compatible with B2. This run starts fresh from ImageNet.  
> If the kernel dies mid-run, just **Run All** again — resumes from HuggingFace automatically.

### Before running
1. Accelerator → **GPU T4 x2**
2. **Add-ons → Secrets** → add secret named `HF_TOKEN` (your HuggingFace write token)
3. In Cell 3: set `HF_REPO = "your-hf-username/coav-contrail-checkpoints"`
4. **Run All**
5. After completion: download `contrail_unet_best.pt` from HuggingFace repo or Output tab

In [ ]:
# ── 1. GPU check ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn

N_GPU = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__}  |  GPUs available: {N_GPU}')
for i in range(N_GPU):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB')

if N_GPU == 0:
    raise RuntimeError('No GPU. Accelerator → GPU T4 x2.')

In [ ]:
# ── 2. Install ────────────────────────────────────────────────────────────────
!pip install segmentation-models-pytorch albumentations huggingface_hub -q
print('Done.')

In [ ]:
# ── 3. Config & paths ─────────────────────────────────────────────────────────
import os

ENCODER    = 'efficientnet-b2'
EPOCHS     = 60     # warm restart: continue from ep 38 to 60 (22 new epochs)
BATCH_SIZE = 8
IMG_SIZE   = 512
START_LR   = 1e-4   # warm restart — fresh LR cycle from high value
WR_EPOCHS  = 22     # length of new cosine cycle (60 - 38 = 22)

HF_REPO        = 'janisdombr/coav-contrail-checkpoints'
HF_PUSH_EVERY  = 1

WORK_DIR = '/kaggle/working'
DATA_DIR = f'{WORK_DIR}/GVCCS/train'
SAVE_DIR = WORK_DIR

PRETRAIN_CKPT = None
working_ckpt  = f'{WORK_DIR}/checkpoint_last.pt'
if os.path.exists(working_ckpt):
    PRETRAIN_CKPT = working_ckpt
    print(f'Local checkpoint found: {PRETRAIN_CKPT}')
else:
    print('No local checkpoint — will restore from HuggingFace in next cell')

print(f'Encoder: {ENCODER}  |  Batch: {BATCH_SIZE}  |  Epochs: {EPOCHS}')
print(f'Warm restart: LR={START_LR:.1e}  cycle={WR_EPOCHS} epochs')

In [ ]:
# ── 3b. HuggingFace Hub setup ─────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download
import shutil

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
hf_api   = HfApi(token=HF_TOKEN)

# Create private repo if it doesn't exist yet
hf_api.create_repo(repo_id=HF_REPO, repo_type='model', private=True, exist_ok=True)
print(f'HF repo ready → https://huggingface.co/{HF_REPO}')


def push_to_hf(is_best=False):
    """Upload checkpoint_last.pt (and contrail_unet_best.pt when improved) to HF Hub."""
    files = ['checkpoint_last.pt']
    if is_best:
        files.append('contrail_unet_best.pt')
    for fname in files:
        path = f'{WORK_DIR}/{fname}'
        if os.path.exists(path):
            hf_api.upload_file(
                path_or_fileobj=path,
                path_in_repo=fname,
                repo_id=HF_REPO,
                repo_type='model',
            )
    pushed = ', '.join(files)
    print(f'[HF] Pushed: {pushed}')


# Restore from HF if no local checkpoint
if PRETRAIN_CKPT is None:
    try:
        local = hf_hub_download(
            repo_id=HF_REPO,
            filename='checkpoint_last.pt',
            repo_type='model',
            token=HF_TOKEN,
            local_dir=WORK_DIR,
        )
        PRETRAIN_CKPT = local
        print(f'[HF] Checkpoint restored from HuggingFace → resuming training')
    except Exception as e:
        print(f'[HF] No checkpoint on HF yet — fresh start from ImageNet weights')

In [ ]:
# ── 4. Download GVCCS (~2.1 GB) ───────────────────────────────────────────────
import json

if not os.path.exists(DATA_DIR):
    print('Downloading GVCCS from Zenodo...')
    !wget -q --show-progress \
        'https://zenodo.org/records/15743988/files/GVCCS.zip?download=1' \
        -O {WORK_DIR}/GVCCS.zip
    print('Extracting...')
    !unzip -q {WORK_DIR}/GVCCS.zip -d {WORK_DIR}/
    !rm {WORK_DIR}/GVCCS.zip
    print('Done.')
else:
    print('GVCCS already present.')

with open(f'{DATA_DIR}/annotations.json') as f:
    _meta = json.load(f)
print(f'Images: {len(_meta["images"])}  |  Annotations: {len(_meta["annotations"])}')

In [ ]:
# ── 5. Dataset ────────────────────────────────────────────────────────────────
import json, cv2, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class GVCCSDataset(Dataset):
    def __init__(self, data_dir, transform=None, indices=None):
        self.data_dir  = Path(data_dir)
        self.transform = transform
        with open(self.data_dir / 'annotations.json') as f:
            coco = json.load(f)
        self.ann_by_img = {}
        for ann in coco['annotations']:
            self.ann_by_img.setdefault(ann['image_id'], []).append(ann)
        imgs = coco['images']
        self.images = [imgs[i] for i in indices] if indices is not None else imgs

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        meta  = self.images[idx]
        image = cv2.cvtColor(
            cv2.imread(str(self.data_dir / 'images' / meta['file_name'])),
            cv2.COLOR_BGR2RGB)
        h, w  = meta.get('height', 1024), meta.get('width', 1024)
        mask  = np.zeros((h, w), dtype=np.uint8)
        for ann in self.ann_by_img.get(meta['id'], []):
            for seg in ann.get('segmentation', []):
                cv2.fillPoly(mask, [np.array(seg, dtype=np.int32).reshape(-1, 2)], 1)
        if self.transform:
            r = self.transform(image=image, mask=mask)
            return r['image'], r['mask'].unsqueeze(0).float()
        return image, mask

print('GVCCSDataset ready.')

In [ ]:
# ── 6. Transforms & DataLoaders ───────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2
import random

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# albumentations v2 API (Kaggle now ships v2 — old parameter names raise warnings)
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    # ── Geometric ──
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.9, 1.1),   # replaces ShiftScaleRotate
             rotate=(-45, 45), p=0.5),
    A.ElasticTransform(alpha=80, sigma=4.0, p=0.25),    # v2: no alpha_affine param
    # ── Photometric ──
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
    A.GaussianBlur(blur_limit=3, p=0.15),
    A.GaussNoise(p=0.2),                                # v2: no var_limit param
    # ── Occlusion ──
    A.CoarseDropout(num_holes_range=(1, 6),             # v2 API
                    hole_height_range=(8, 48),
                    hole_width_range=(8, 48), p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

with open(f'{DATA_DIR}/annotations.json') as f:
    _n = len(json.load(f)['images'])

# GVCCS images are ordered by video sequence (consecutive frames).
# Taking last 10% = only the last ~12 sequences → biased val set → oscillating Dice.
# Fix: shuffle with fixed seed so val images come from ALL 122 sequences uniformly.
random.seed(42)
all_indices = list(range(_n))
random.shuffle(all_indices)
val_size  = int(_n * 0.1)
val_idx   = all_indices[:val_size]
train_idx = all_indices[val_size:]

train_loader = DataLoader(
    GVCCSDataset(DATA_DIR, train_tf, train_idx),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=False)
val_loader = DataLoader(
    GVCCSDataset(DATA_DIR, val_tf, val_idx),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)

print(f'Train: {len(train_loader.dataset)} images ({len(train_loader)} batches)')
print(f'Val:   {len(val_loader.dataset)} images ({len(val_loader)} batches) — shuffled, seed=42')

In [ ]:
# ── 7. Model ──────────────────────────────────────────────────────────────────
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,
)

start_epoch = 1
best_dice   = 0.0
history     = []

if PRETRAIN_CKPT:
    ckpt = torch.load(PRETRAIN_CKPT, map_location='cpu')
    if isinstance(ckpt, dict) and 'model_state' in ckpt:
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt.get('epoch', 0) + 1
        best_dice   = ckpt.get('best_dice', 0.0)
        history     = ckpt.get('history', [])
        print(f'Resumed from epoch {start_epoch - 1}, best dice {best_dice:.4f}')
    else:
        print('Checkpoint format not recognised — starting from epoch 1')
else:
    print('Fresh start from ImageNet weights (epoch 1)')

# Single GPU — DataParallel causes CUDA misaligned address on PyTorch 2.x + albumentations v2
model = model.cuda()

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Encoder: {ENCODER}  |  Parameters: {params:.1f}M  |  Start epoch: {start_epoch}')
print('Mode: single GPU (T4 15 GB — B2 + batch 8 fits easily)')

In [ ]:
# ── 8. Loss & optimizer ───────────────────────────────────────────────────────
dice_loss  = smp.losses.DiceLoss(mode='binary')
focal_loss = smp.losses.FocalLoss(mode='binary', gamma=2.0, alpha=0.25)
loss_fn    = lambda p, t: dice_loss(p, t) + focal_loss(p, t)

optimizer = torch.optim.AdamW(model.parameters(), lr=START_LR, weight_decay=2e-4)

# Warm restart: fresh cosine cycle of length WR_EPOCHS starting at START_LR.
# No scheduler fast-forward — optimizer must begin at full LR to escape the plateau.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=WR_EPOCHS, eta_min=1e-6)

print(f'Loss: Dice + FocalLoss(gamma=2, alpha=0.25)')
print(f'Warm restart LR: {START_LR:.1e} → 1e-6 over {WR_EPOCHS} epochs')

In [ ]:
# ── 9. Training loop ──────────────────────────────────────────────────────────
from tqdm.notebook import tqdm
import time


def dice_score(logits, target, threshold=0.5):
    pred  = (torch.sigmoid(logits) > threshold).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum()
    if union < 1:
        return 1.0
    return (2 * inter / (union + 1e-8)).item()


def save_checkpoint(epoch, is_best=False):
    ckpt = {
        'epoch':           epoch,
        'encoder':         ENCODER,
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_dice':       best_dice,
        'history':         history,
    }
    torch.save(ckpt, f'{SAVE_DIR}/checkpoint_last.pt')
    if is_best:
        torch.save(model.state_dict(), f'{SAVE_DIR}/contrail_unet_best.pt')


try:
    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()

        model.train()
        tr_loss = tr_dice = 0.0
        for images, masks in tqdm(train_loader,
                                  desc=f'Ep {epoch:02d}/{EPOCHS} train', leave=False):
            images, masks = images.cuda(), masks.cuda()
            optimizer.zero_grad()
            logits = model(images)
            loss   = loss_fn(logits, masks)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item()
            tr_dice += dice_score(logits, masks)

        model.eval()
        vl_loss = vl_dice = 0.0
        with torch.no_grad():
            for images, masks in tqdm(val_loader,
                                      desc=f'Ep {epoch:02d}/{EPOCHS} val  ', leave=False):
                images, masks = images.cuda(), masks.cuda()
                logits = model(images)
                vl_loss += loss_fn(logits, masks).item()
                vl_dice += dice_score(logits, masks)

        scheduler.step()

        nb, nvb = len(train_loader), len(val_loader)
        ep_dice = vl_dice / nvb
        is_best = ep_dice > best_dice
        if is_best:
            best_dice = ep_dice

        row = dict(epoch=epoch,
                   tr_loss=tr_loss/nb, tr_dice=tr_dice/nb,
                   vl_loss=vl_loss/nvb, vl_dice=ep_dice)
        history.append(row)

        # Save locally
        save_checkpoint(epoch, is_best=is_best)

        # Push to HuggingFace Hub
        if epoch % HF_PUSH_EVERY == 0:
            try:
                push_to_hf(is_best=is_best)
            except Exception as hf_err:
                print(f'[HF] Push failed (training continues): {hf_err}')

        marker = '  ← best' if is_best else ''
        print(f'Ep {epoch:02d}/{EPOCHS}  '
              f'tr loss={row["tr_loss"]:.4f} dice={row["tr_dice"]:.4f}  '
              f'val loss={row["vl_loss"]:.4f} dice={ep_dice:.4f}  '
              f'{time.time()-t0:.0f}s{marker}')

except KeyboardInterrupt:
    print('\nInterrupted — saving checkpoint...')
    save_checkpoint(epoch)
    try:
        push_to_hf(is_best=False)
    except Exception as hf_err:
        print(f'[HF] Push on interrupt failed: {hf_err}')

print(f'\nDone. Best val Dice: {best_dice:.4f}')
print(f'Weights saved to HuggingFace: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── 10. Training curves ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_x = [r['epoch'] for r in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_x, [r['tr_loss'] for r in history], label='train')
ax1.plot(epochs_x, [r['vl_loss'] for r in history], label='val')
ax1.set_xlabel('Epoch'); ax1.set_title('Loss (Dice + Focal)'); ax1.legend()

ax2.plot(epochs_x, [r['tr_dice'] for r in history], label='train')
ax2.plot(epochs_x, [r['vl_dice'] for r in history], label='val')
ax2.axhline(0.60, color='orange', linestyle='--', label='demo threshold 0.60')
ax2.axhline(0.75, color='green',  linestyle='--', label='PoC threshold 0.75')
ax2.set_xlabel('Epoch'); ax2.set_title('Dice score'); ax2.legend()

# Annotate V1 best for comparison
ax2.axhline(0.4918, color='grey', linestyle=':', alpha=0.6, label='V1 (B4) best 0.49')
ax2.legend()

plt.suptitle(f'V2 ({ENCODER}) — Best val Dice = {best_dice:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves_v2.png', dpi=120)
plt.show()

In [ ]:
# ── 11. Visual predictions on 6 val images ────────────────────────────────────
import random

raw_model = model.module if hasattr(model, 'module') else model
raw_model.eval()

with open(f'{DATA_DIR}/annotations.json') as f:
    all_imgs = json.load(f)['images']

samples = random.sample(val_idx, 6)
fig, axes = plt.subplots(6, 3, figsize=(11, 22))

for row, idx in enumerate(samples):
    meta = all_imgs[idx]
    img  = cv2.cvtColor(
        cv2.imread(f'{DATA_DIR}/images/{meta["file_name"]}'),
        cv2.COLOR_BGR2RGB)
    inp  = val_tf(image=img)['image'].unsqueeze(0).cuda()
    with torch.no_grad():
        pred = torch.sigmoid(raw_model(inp))[0, 0].cpu().numpy()

    axes[row, 0].imshow(img);               axes[row, 0].set_title('Input')
    axes[row, 1].imshow(pred, cmap='hot');  axes[row, 1].set_title('Heatmap')
    axes[row, 2].imshow(pred > 0.5, cmap='gray'); axes[row, 2].set_title('Mask (t=0.5)')
    for ax in axes[row]: ax.axis('off')

plt.suptitle(f'V2 val samples — Best Dice {best_dice:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/sample_predictions_v2.png', dpi=100)
plt.show()

In [ ]:
# ── 12. Output summary ────────────────────────────────────────────────────────
import os
print(f'Encoder:    {ENCODER}')
print(f'Best dice:  {best_dice:.4f}  (V1 baseline: 0.4918)')
print(f'Improvement: {(best_dice - 0.4918):+.4f}')
print()
for fname in ['contrail_unet_best.pt', 'checkpoint_last.pt',
              'training_curves_v2.png', 'sample_predictions_v2.png']:
    path = f'{SAVE_DIR}/{fname}'
    size = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    print(f'  {fname:<40} {size:.1f} MB' if size else f'  {fname:<40} MISSING')

print(f"""
HuggingFace repo: https://huggingface.co/{HF_REPO}

Download contrail_unet_best.pt from HF or Output tab, then:
  → edge-pi/python/weights/contrail_unet.pt

Thresholds:
  Dice > 0.60  → demo-ready
  Dice > 0.75  → production PoC
""")